# SQL Server → Snowflake generator

Generates everything needed to land a SQL Server source into Snowflake and model it in dbt:

1. **Master connector config** : registers the connection in `TelTools_SQLServer_Connectors.json`
2. **Extract config** : `TelTools_SQLServer_<CONN>_Extract.json` (the DataSetN list)
3. **dbt source YAML** : `_src/<source>.yml`
4. **dbt bronze models** : one `.sql` per table (shreds `JSONFIELD` → typed columns)
5. **Snowflake landing DDL** : one `JSONFIELD VARIANT` table per source table

All the logic lives in **`sqlserver_to_snowflake.py`** (keep it next to this notebook). You only edit the **one config cell** below. Every step is **idempotent**. re-running never appends to or corrupts a config file; configs are merged in memory by identity and rewritten whole.

## 0 · Setup

`pip install pyodbc pyyaml` once if needed. Then edit the single config below — this is the **only** place you change things per source.

In [ ]:
import sys
sys.path.insert(0, r"C:\Sources\Repos\EnterpriseCloud\astro_datautility\include\Python")
from TELTools import getAzureSecret

PASSWORD_NAME = "TELQuoteSystem-Password"

PASSWORD = getAzureSecret(PASSWORD_NAME).value

USERNAME = getAzureSecret("TELQuoteSystem-Username").value 

print("USERNAME:", USERNAME)
print("PASSWORD:", PASSWORD)


USERNAME: teh_ProcessOrder_reader
PASSWORD_NAME: TELQuoteSystem-Password
PASSWORD: rYMEs{w(7U


In [14]:
from pathlib import Path
import sqlserver_to_snowflake as s

cfg = s.PipelineConfig(
    server          = "teloutdbp001.asia.tel.com",
    database        = "outsystems",
    username        = USERNAME,
    password        = PASSWORD,
    password_name   = PASSWORD_NAME,

    connection_name = "TELQS",
    landing_schema  = "TELQS",
    dbt_subfolder   = "TEA_SERVICE",
    bronze_tests_filename = "_models.yml",
    landing_db      = "LZ_DEV",
    dbt_project     = "dbt_tel",

    model_prefix    = "TELQS",
    model_alias_prefix    = "TELQS",

    bronze_db = "BRONZE_DEV" ,
    bronze_schema = "TEA_SERVICE",

    snow_ticket     = "K.44312",
    ingestion_label = "TELQS_tables",
    lz_role         = "SYSADMIN_DEV",
    bronze_role     = "PLATFORM_TEAM_DEV",

    repo_root       = Path(r"C:\Sources\Repos\EnterpriseCloud"),

    tables = [
    ("outsystems", "dbo", "OSUSR_69G_ENGINEERONSITE"), 
    ("outsystems", "dbo", "OSUSR_BN7_ADVANCEFILTERDATACONTRACTTYPE"),
    ("outsystems", "dbo", "OSUSR_BN7_SQCPROCESSORDERPRICING"),
    ("outsystems", "dbo", "OSUSR_BN7_VERSIONHISTORY"),
    ("outsystems", "dbo", "OSUSR_CZN_CONTRACTTYPE")
],
)

print("connection      :", cfg.connection_name)
print("alias prefix    :", cfg.alias_prefix)
print("bronze prefix   :", cfg.bronze_prefix)
print("source yaml     :", cfg.source_yaml_file)
print("bronze path     :", cfg.bronze_path)
print("extract config  :", cfg.extract_config_file)
print("master config   :", cfg.master_config_file)
print("lz snowfile     :", cfg.lz_snow_file.name)
print("bronze snowfile :", cfg.bronze_snow_file.name)


connection      : TELQS
alias prefix    : TELQS
bronze prefix   : BRONZE__TEA_SERVICE__TELQS
source yaml     : C:\Sources\Repos\EnterpriseCloud\astro_datautility\dags\dbt\dbt_tel\models\_src\telqs.yml
bronze path     : C:\Sources\Repos\EnterpriseCloud\astro_datautility\dags\dbt\dbt_tel\models\tea_service\bronze
extract config  : C:\Sources\Repos\EnterpriseCloud\astro_datautility\include\Configuration\TelTools_SQLServer_TELQS_Extract.json
master config   : C:\Sources\Repos\EnterpriseCloud\astro_datautility\include\Configuration\TelTools_SQLServer_Connectors.json
lz snowfile     : K.44312__1.lz_TELQS_tables__data.SQL
bronze snowfile : K.44312__2.bronze_TELQS_tables__data.SQL


## 1 · Introspect SQL Server

Opens **one** connection, checks every table exists, reads its columns (ordered), and maps each type to Snowflake. Set `with_counts=True` if you also want row counts (slower).

In [15]:
with s.SqlServerInspector(cfg) as inspector:
    metas = inspector.inspect(with_counts=True)

for m in metas:
    print(f"[{m.database}].[{m.schema}].[{m.table}]  ->  {len(m.columns)} columns ->  {m.row_count} rows")


[outsystems].[dbo].[OSUSR_69G_ENGINEERONSITE]  ->  14 columns ->  420 rows
[outsystems].[dbo].[OSUSR_BN7_ADVANCEFILTERDATACONTRACTTYPE]  ->  3 columns ->  1 rows
[outsystems].[dbo].[OSUSR_BN7_SQCPROCESSORDERPRICING]  ->  10 columns ->  1117 rows
[outsystems].[dbo].[OSUSR_BN7_VERSIONHISTORY]  ->  24 columns ->  7175 rows
[outsystems].[dbo].[OSUSR_CZN_CONTRACTTYPE]  ->  5 columns ->  26 rows


Optional — peek at the column → Snowflake type mapping for one table to confirm the types look right before generating anything.

In [16]:
first = metas[0]
for col, sf_type in first.columns.items():
    print(f"{col:<40} {sf_type}")


ID                                       NUMBER(38,2)
SERVICEQUOTEWITHCONTRACTID               NUMBER(38,2)
MACHINESID                               VARCHAR
REQUESTTOFSC                             VARCHAR
ISNEWCONTRACT                            BOOLEAN
ISWARRANTY                               BOOLEAN
ISNEEDS_A_BILLING                        BOOLEAN
IS_ACTIVE                                BOOLEAN
CREADTEDBY                               NUMBER(38,2)
UPDATEDBY                                NUMBER(38,2)
CREATEDON                                TIMESTAMP_NTZ
UPDATEDON                                TIMESTAMP_NTZ
ENGINEEROTHERDETAILID                    NUMBER(38,2)
IS_BILLINGMATRIXWITHOUTQUOTE             BOOLEAN


## 2 · Write the TelTools configs

- **Master connector**: merges this connection into the file by name (other connections preserved).
- **Extract config**: `mode="merge"` upserts datasets by `(schema, table)` and renumbers `DataSet1..N`; use `mode="replace"` to make the current table list the whole block. Either way the file is rebuilt and overwritten **no append bug**.

In [ ]:
print("wrote:", s.write_master_connector(cfg))
print("wrote:", s.write_extract_config(cfg, metas, mode="insert_overwrite"))


## 3 · Write the dbt artifacts

- **Source YAML**: upserts tables under this source by name; other sources in the file are kept.
- **Bronze models**: one `.sql` per table → `models/<subfolder>/bronze/`.

In [ ]:
print("wrote source yaml:", s.write_source_yaml(cfg, metas))

for p in s.write_bronze_models(cfg, metas):
    print("wrote model:", p.name)


### Bronze dbt tests

Writes a schema.yml into the **bronze** folder attaching the generic tests 
(`has_data`, `recently_run` by default) to every bronze model. 
Same idempotent merge as the source YAML: re-running upserts by model name, 
never duplicates. Override which tests to attach via `bronze_tests=[...]` on the config.

In [ ]:
print("wrote bronze tests:", s.write_bronze_tests_yaml(cfg, metas))

print(cfg.bronze_tests_yaml_file.read_text())


Preview one generated bronze model to eyeball it.

In [ ]:
print(s.bronze_model(metas[0], cfg))


## 4 · Write the Snowflake snowfiles

Two snowfiles are written to `<repo>/Snow/`, each with a `USE ROLE` header and env-templated tokens. Landing uses `<<LZ>>`/`<<SYSADMIN>>`; bronze uses `<<BRONZE>>`/`<<PLATFORM_TEAM>>` (`LZ_DEV`→`<<LZ>>`, `BRONZE_DEV`→`<<BRONZE>>`, `SYSADMIN_DEV`→`<<SYSADMIN>>`, `PLATFORM_TEAM_DEV`→`<<PLATFORM_TEAM>>`):

- **LZ** (`USE ROLE SYSADMIN_DEV`): raw `JSONFIELD VARIANT` landing tables.
- **Bronze** (`USE ROLE PLATFORM_TEAM_DEV`): typed tables named after the dbt alias (`<schema>_<table>`), in the `<<BRONZE>>` database.

In [ ]:
lz_path = s.write_lz_ddl(cfg, metas)
bz_path = s.write_bronze_ddl(cfg, metas)
print("wrote LZ     :", lz_path.name)
print("wrote bronze :", bz_path.name)

print("\n----- LZ (first table) -----")
print(s.apply_env_tokens(f"USE ROLE {cfg.lz_role};\n\n" + s.landing_ddl(metas[0], cfg), cfg))
print("\n----- BRONZE (first table) -----")
print(s.apply_env_tokens(f"USE ROLE {cfg.bronze_role};\n\n" + s.bronze_ddl(metas[0], cfg), cfg, layer="bronze"))


## 4b · Run the snowfiles on Snowflake

Reads each generated `.SQL` file, substitutes the `<<...>>` tokens with the real values in `SNOW_VARS`, and runs it on Snowflake. The generator emits four tokens: `<<LZ>>`, `<<BRONZE>>`, `<<SYSADMIN>>`, `<<PLATFORM_TEAM>>`.

Keep `DRY_RUN = True` to just print the rendered SQL; set it `False` to execute (needs `pip install snowflake-connector-python` and valid `SNOW_CONN` credentials). Any token missing from `SNOW_VARS` is flagged before anything runs.

In [ ]:
SNOW_VARS = {
    "<<LZ>>":            "LZ_DEV",
    "<<BRONZE>>":        "BRONZE_DEV",
    "<<SYSADMIN>>":      "PLATFORM_TEAM_DEV",
    "<<PLATFORM_TEAM>>": "PLATFORM_TEAM_DEV",
}

SNOW_CONN = dict(
    account   = "telus-global.privatelink",
    user = "stanley.x.correia@us.tel.com",
    authenticator="externalbrowser",
    role      = "PLATFORM_TEAM_DEV",
    warehouse = "DEV_WH",
    database  = "LZ_DEV",
    schema    = cfg.landing_schema.upper(),
)

DRY_RUN = False

In [ ]:
import re as _re

def render_tokens(text):
    for tok, val in SNOW_VARS.items():
        text = text.replace(tok, val)
    leftover = sorted(set(_re.findall(r"<<[^>]+>>", text)))
    return text, leftover

def run_snowfile(path, conn):
    sql, leftover = render_tokens(Path(path).read_text(encoding="utf-8"))
    print("=" * 70)
    print(Path(path).name)
    print("=" * 70)
    if leftover:
        print("UNRESOLVED TOKENS (add to SNOW_VARS):", ", ".join(leftover))
    if DRY_RUN or conn is None:
        print(sql)
        return
    for i, cur in enumerate(conn.execute_string(sql), start=1):
        print(f"  stmt {i}: ok  sfqid={cur.sfqid}  rows={cur.rowcount}")

conn = None
if not DRY_RUN:
    import snowflake.connector
    conn = snowflake.connector.connect(**{k: v for k, v in SNOW_CONN.items() if v != ""})

try:
    run_snowfile(lz_path, conn)
    run_snowfile(bz_path, conn)
finally:
    if conn is not None:
        conn.close()
        print("\nsnowflake connection closed.")

## 5 · Run the extractor (optional)

Build the per-table extract commands, review them, then run. Each command is shown with its return code and any output/errors.

In [ ]:
commands = s.build_extract_commands(cfg, metas)
commands


In [ ]:
import subprocess

working_directory = str(cfg.repo_root / "astro_datautility")
for cmd in commands:
    result = subprocess.run(cmd, cwd=working_directory, capture_output=True, text=True)
    print(f"$ {cmd}")
    print(f"  return code: {result.returncode}")
    if result.stdout:
        print(f"  stdout:\n{result.stdout}")
    if result.stderr:
        print(f"  stderr:\n{result.stderr}")
